# Indexação e busca híbrida com um benchmark em português

Este tutorial utiliza o conjunto **Pirá 2.0**, formado por perguntas e textos de apoio sobre o oceano, a costa brasileira e mudanças climáticas. Os exemplos são baixados do Hugging Face durante a execução.

Os documentos passam pelo MinIO e pelo fluxo de indexação do WebSensors Flow RAG. As consultas utilizam os endpoints da API já em execução. São comparadas a busca BM25, a busca semântica por kNN e a busca híbrida que combina os resultados com RRF na aplicação.

O experimento trabalha com **no máximo 500 documentos**. Por padrão utiliza um recorte menor para facilitar a execução local. As perguntas não são inseridas nos documentos indexados. Isso evita um vazamento direto do texto da consulta para o conteúdo pesquisável.

Fonte do conjunto de dados no [Hugging Face](https://huggingface.co/datasets/paulopirozelli/pira). O conjunto está distribuído sob licença CC BY 4.0.

## 1. Parâmetros do experimento

O serviço da API e o MinIO já devem estar disponíveis. As credenciais do MinIO e o token opcional da API são lidos do YAML global do projeto. O endereço externo do MinIO pode diferir do endereço utilizado entre contêineres.

A partição de teste contém 227 pares de pergunta e resposta. Cada documento será formado somente pelo texto de apoio em português. Os textos repetidos serão indexados uma única vez.

In [3]:
import hashlib
import io
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse

import httpx
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display
from minio import Minio
from tqdm.auto import tqdm

API_URL = "http://127.0.0.1:8765" # confirmar
MINIO_URL = "http://127.0.0.1:9000"
DIVISAO = "test"
MAX_DOCUMENTOS = 120
MAX_CONSULTAS = 30
TOP_K_API = 30
K_AVALIACAO = 10
PARALELISMO_INDEXACAO = 3
INTERVALO_POLL = 2.0
LIMITE_SEGUNDOS_POR_LOTE = 1800
SEMENTE = 42

assert DIVISAO in {"train", "validation", "test"}
assert 1 <= MAX_DOCUMENTOS <= 500
assert 1 <= MAX_CONSULTAS <= 500
assert 1 <= K_AVALIACAO <= TOP_K_API <= 50
assert 1 <= PARALELISMO_INDEXACAO <= 8

pd.set_option("display.max_colwidth", 130)
random.seed(SEMENTE)

## 2. Conexão com o YAML e a API

O notebook localiza o arquivo de configuração no diretório do projeto ou no diretório acima. A verificação de acesso usa a rota de descrição do serviço e não executa instalação nem inicialização.

In [ ]:
CANDIDATOS_YAML = [
    Path("../config/rag.local.yaml"),
]
CAMINHO_YAML = next((p.resolve() for p in CANDIDATOS_YAML if p.exists()), None)
if CAMINHO_YAML is None:
    raise FileNotFoundError("O YAML do serviço não foi encontrado no projeto")

CONFIG = yaml.safe_load(CAMINHO_YAML.read_text(encoding="utf-8"))
autenticacao = CONFIG.get("api", {}).get("auth", {})
CABECALHOS = (
    {"Authorization": f"Bearer {autenticacao['token']}"}
    if autenticacao.get("enabled") and autenticacao.get("token")
    else {}
)
api = httpx.Client(base_url=API_URL.rstrip("/"), headers=CABECALHOS, timeout=90.0)

minio_cfg = CONFIG["services"]["minio"]
minio_endereco = urlparse(MINIO_URL)
minio = Minio(
    minio_endereco.netloc,
    access_key=minio_cfg["access_key"],
    secret_key=minio_cfg["secret_key"],
    secure=minio_endereco.scheme == "https",
    region=minio_cfg.get("region"),
)
BUCKET_BRONZE = minio_cfg["bronze_bucket"]
BUCKET_SILVER = minio_cfg["silver_bucket"]
COLECAO = f"pira2_{DIVISAO}"
TOP_K_API = min(TOP_K_API, int(CONFIG.get("search", {}).get("max_top_k", 50)))
K_AVALIACAO = min(K_AVALIACAO, TOP_K_API)
PREFIXO = f"benchmarks/pira2/{DIVISAO}"


def chamar_api(metodo, rota, corpo=None, parametros=None):
    resposta = api.request(metodo, rota, json=corpo, params=parametros)
    if resposta.is_error:
        raise RuntimeError(f"Falha HTTP {resposta.status_code} em {rota}\n{resposta.text[:1500]}")
    return resposta.json()

info_servico = chamar_api("GET", "/rag")
print("Serviço conectado", info_servico.get("name"))
print("Coleção do experimento", COLECAO)
print("Bucket de entrada", BUCKET_BRONZE)
print("Limite de documentos", MAX_DOCUMENTOS)

ConnectError: [Errno 111] Connection refused

## 3. Download do Pirá 2.0

A API pública do Hugging Face permite obter somente as linhas necessárias. Os arquivos retornam pares de perguntas e seus textos de apoio, sem necessidade de baixar todo o conjunto de treinamento.

O download é gravado em um cache local. Ao executar novamente o notebook, os dados podem ser reutilizados sem outra transferência.

In [ ]:
CACHE = Path(".cache/pira2")
CACHE.mkdir(parents=True, exist_ok=True)
CAMINHO_CACHE = CACHE / f"{DIVISAO}.json"
URL_DATASET = "https://datasets-server.huggingface.co/rows"


def baixar_linhas_pira(divisao, usar_cache=True):
    if usar_cache and CAMINHO_CACHE.exists():
        return json.loads(CAMINHO_CACHE.read_text(encoding="utf-8"))

    registros = []
    total = None
    with httpx.Client(timeout=90.0, follow_redirects=True) as cliente:
        for deslocamento in range(0, 100000, 100):
            parametros = {
                "dataset": "paulopirozelli/pira",
                "config": "default",
                "split": divisao,
                "offset": deslocamento,
                "length": 100,
            }
            for tentativa in range(4):
                resposta = cliente.get(URL_DATASET, params=parametros)
                if resposta.status_code not in (429, 500, 502, 503, 504):
                    break
                time.sleep(2 ** tentativa)
            resposta.raise_for_status()
            lote = resposta.json()
            total = lote.get("num_rows_total", total)
            registros.extend(item["row"] for item in lote.get("rows", []))
            print(f"Download concluído parcialmente  {len(registros)} de {total or '?'} exemplos")
            if not lote.get("rows") or (total is not None and len(registros) >= total):
                break
    if not registros:
        raise RuntimeError("O conjunto não retornou registros")
    CAMINHO_CACHE.write_text(json.dumps(registros, ensure_ascii=False), encoding="utf-8")
    return registros

LINHAS_BRUTAS = baixar_linhas_pira(DIVISAO)
print(f"Exemplos recebidos  {len(LINHAS_BRUTAS)}")
print("Campos disponíveis", sorted(LINHAS_BRUTAS[0].keys()))

## 4. Construção da coleção documental

Um documento representa um texto de apoio único. O identificador é derivado do conteúdo e permanece estável em execuções repetidas. A pergunta original será mantida somente no conjunto de avaliação e nunca será adicionada ao Markdown do documento.

Entradas sem pergunta ou sem texto em português são ignoradas. Textos iguais são agrupados e podem ter várias perguntas associadas. A coleção fica limitada ao valor escolhido em `MAX_DOCUMENTOS`.

In [ ]:
def texto_util(valor):
    return isinstance(valor, str) and len(valor.strip()) >= 1


def identificador_documento(texto):
    digest = hashlib.sha256(texto.encode("utf-8")).hexdigest()[:20]
    return f"pira2-{digest}"


def preparar_colecao(linhas, limite):
    documentos = {}
    perguntas = []
    for registro in linhas:
        texto = registro.get("abstract_translated_pt")
        pergunta = registro.get("question_pt_origin")
        resposta_presente = registro.get("answer_in_text")
        if resposta_presente is False or resposta_presente == 0:
            continue
        if not texto_util(texto) or not texto_util(pergunta):
            continue
        texto, pergunta = texto.strip(), pergunta.strip()
        if len(texto) < 180 or len(pergunta) < 12:
            continue
        # O texto de apoio é a única fonte documental indexada.
        doc_id = identificador_documento(texto)
        if doc_id not in documentos and len(documentos) >= limite:
            continue
        documentos.setdefault(doc_id, {
            "document_id": doc_id,
            "texto": texto,
            "titulo": f"Texto de apoio Pirá {len(documentos) + 1:03d}",
            "origem": str(registro.get("id_qa") or ""),
        })
        # O gabarito de recuperação é a associação entre pergunta e texto de apoio.
        perguntas.append({
            "query_id": str(registro.get("id_qa") or f"q-{len(perguntas)+1}"),
            "query": pergunta,
            "document_id": doc_id,
            "resposta_referencia": registro.get("answer_pt_origin") or "",
        })
    return documentos, perguntas

DOCUMENTOS, PERGUNTAS = preparar_colecao(LINHAS_BRUTAS, MAX_DOCUMENTOS)
if not DOCUMENTOS or not PERGUNTAS:
    raise RuntimeError("Não foram encontrados textos de apoio em português. Confira o schema do conjunto")
assert len(DOCUMENTOS) <= 500
PERGUNTAS_AVALIACAO = random.Random(SEMENTE).sample(
    PERGUNTAS, min(MAX_CONSULTAS, len(PERGUNTAS))
)

print("Documentos únicos para indexação", len(DOCUMENTOS))
print("Perguntas associadas à coleção", len(PERGUNTAS))
print("Perguntas selecionadas para avaliação", len(PERGUNTAS_AVALIACAO))
display(pd.DataFrame(PERGUNTAS_AVALIACAO)[["query_id", "query", "document_id"]].head(8))

## 5. Inspeção de um documento

O conteúdo em Markdown será composto pelo texto de apoio e por um título neutro. Perguntas e respostas de referência não são incluídas no conteúdo indexado, pois isso facilitaria artificialmente a recuperação.

In [ ]:
DOC_EXEMPLO = next(iter(DOCUMENTOS.values()))


def gerar_markdown(documento):
    return f"# {documento['titulo']}\n\n{documento['texto']}\n"

print(gerar_markdown(DOC_EXEMPLO)[:2200])

## 6. Envio dos documentos ao MinIO Bronze

Cada texto é salvo como um arquivo Markdown individual. Os nomes utilizam identificadores estáveis para permitir novas execuções sem criar cópias adicionais.

Esta operação ainda não indexa os documentos. Ela apenas disponibiliza os arquivos originais para o fluxo de ingestão.

In [ ]:
CHAVES = {}
for documento in tqdm(DOCUMENTOS.values(), desc="Enviando documentos", unit="arquivo"):
    chave = f"{PREFIXO}/{documento['document_id']}.md"
    conteudo = gerar_markdown(documento).encode("utf-8")
    minio.put_object(
        BUCKET_BRONZE,
        chave,
        io.BytesIO(conteudo),
        length=len(conteudo),
        content_type="text/markdown; charset=utf-8",
    )
    CHAVES[documento["document_id"]] = chave
print(f"Arquivos disponíveis no MinIO Bronze  {len(CHAVES)}")

## 7. Primeira indexação com acompanhamento dos steps

O endpoint de indexação é assíncrono. Ele retorna um token para consultar o progresso. A saída apresenta o step em execução, o tempo decorrido e o ETA informado pelo serviço, quando disponível.

Ao terminar, o documento terá Markdown e JSON estrutural no MinIO Silver e chunks pesquisáveis no Elasticsearch.

In [ ]:
def iniciar_indexacao(doc_id):
    pedido = {
        "bucket": BUCKET_BRONZE,
        "key": CHAVES[doc_id],
        "document_id": doc_id,
        "metadata": {
            "benchmark": "pira2",
            "colecao": COLECAO,
            "divisao": DIVISAO,
            "idioma": "pt",
        },
    }
    return chamar_api("POST", "/documents/index", pedido)


def consultar_indexacao(token):
    return chamar_api("GET", f"/indexing/{token}")


def acompanhar_token(token, tempo_maximo=1800):
    inicio = time.monotonic()
    ultimo_step = None
    while True:
        atual = consultar_indexacao(token)
        progresso = atual.get("progress") or {}
        step = progresso.get("current_step") or "aguardando"
        if step != ultimo_step:
            print(f"Step  {step}  Estado  {atual['status']}  Decorrido  {time.monotonic()-inicio:.1f}s")
            ultimo_step = step
        if atual.get("terminal"):
            if atual["status"] != "success":
                raise RuntimeError(f"Indexação falhou\n{json.dumps(atual.get('error'), ensure_ascii=False, indent=2)}")
            return atual
        if time.monotonic() - inicio > tempo_maximo:
            raise TimeoutError(f"Tempo limite de acompanhamento do token {token}. O job pode continuar no servidor")
        time.sleep(INTERVALO_POLL)

PRIMEIRO_ID = next(iter(DOCUMENTOS))
primeiro_inicio = iniciar_indexacao(PRIMEIRO_ID)
PRIMEIRO_RESULTADO = acompanhar_token(primeiro_inicio["token"])
print("Resultado", json.dumps(PRIMEIRO_RESULTADO.get("result"), ensure_ascii=False, indent=2)[:1500])

## 8. Indexação do restante da coleção

As próximas solicitações são enviadas em lotes pequenos para respeitar a concorrência disponível. O progresso exibe a quantidade de documentos concluídos e um ETA derivado do tempo observado nesta execução.

A indexação é idempotente. Quando um documento já está atualizado, o serviço pode responder com uma execução sem alterações.

In [ ]:
def indexar_lotes(ids, tamanho_lote=3, intervalo=2.0, limite_por_lote=1800):
    resultados = {}
    falhas = {}
    iniciou = time.monotonic()
    ids = list(ids)
    with tqdm(total=len(ids), desc="Indexação", unit="documento") as barra:
        for inicio_lote in range(0, len(ids), tamanho_lote):
            lote_ids = ids[inicio_lote:inicio_lote + tamanho_lote]
            pendentes = {}
            for doc_id in lote_ids:
                try:
                    dado = iniciar_indexacao(doc_id)
                    pendentes[doc_id] = dado["token"]
                except Exception as erro:
                    falhas[doc_id] = str(erro)
                    barra.update(1)
            limite = time.monotonic() + limite_por_lote
            while pendentes:
                for doc_id, token in list(pendentes.items()):
                    try:
                        retorno = consultar_indexacao(token)
                    except Exception as erro:
                        falhas[doc_id] = f"Não foi possível consultar o token  {erro}"
                        pendentes.pop(doc_id)
                        barra.update(1)
                        continue
                    if retorno.get("terminal"):
                        if retorno["status"] == "success":
                            resultados[doc_id] = retorno
                        else:
                            falhas[doc_id] = retorno.get("error") or "Falha na indexação"
                        pendentes.pop(doc_id)
                        barra.update(1)
                if pendentes:
                    if time.monotonic() >= limite:
                        raise TimeoutError(f"Lote excedeu o tempo limite. Tokens em andamento  {pendentes}")
                    time.sleep(intervalo)
            realizados = len(resultados) + len(falhas)
            decorridos = time.monotonic() - iniciou
            eta = decorridos / realizados * (len(ids) - realizados) if realizados else None
            barra.set_postfix_str(f"concluídos {len(resultados)}  falhas {len(falhas)}  ETA {eta:.0f}s" if eta is not None else "ETA calculando")
    return resultados, falhas

RESTANTES = [doc_id for doc_id in DOCUMENTOS if doc_id != PRIMEIRO_ID]
RESULTADOS_RESTANTES, FALHAS_INDEXACAO = indexar_lotes(
    RESTANTES,
    tamanho_lote=PARALELISMO_INDEXACAO,
    intervalo=INTERVALO_POLL,
    limite_por_lote=LIMITE_SEGUNDOS_POR_LOTE,
)

DOCUMENTOS_OK = set(RESULTADOS_RESTANTES) | {PRIMEIRO_ID}
print("Indexações concluídas", len(DOCUMENTOS_OK))
print("Falhas", len(FALHAS_INDEXACAO))
if FALHAS_INDEXACAO:
    display(pd.DataFrame([{"document_id": k, "erro": str(v)} for k,v in FALHAS_INDEXACAO.items()]).head(10))

## 9. Busca BM25

BM25 ordena passagens com base na correspondência lexical entre os termos da pergunta e o conteúdo indexado. A consulta usa uma pergunta real do Pirá e restringe a busca aos documentos deste experimento.

In [ ]:
ROTAS = {
    "BM25": "/search/bm25",
    "Semântica": "/search/semantic",
    "Híbrida": "/search/hybrid",
}
FILTROS = {"colecao": COLECAO}


def buscar(metodo, pergunta, top_k=TOP_K_API):
    return chamar_api("POST", ROTAS[metodo], {
        "query": pergunta,
        "top_k": top_k,
        "filters": FILTROS,
        "debug": True,
    })


def tabela_busca(resposta, limite=10):
    linhas = []
    for item in resposta.get("results", [])[:limite]:
        linhas.append({
            "posição": item["rank"],
            "document_id": item["document_id"],
            "score": item.get("score"),
            "trecho": (item.get("content") or "").replace("\n", " ")[:180],
        })
    display(pd.DataFrame(linhas))
    print(f"Tempo observado  {resposta.get('took_ms')} ms")

EXEMPLO_QUERY = PERGUNTAS_AVALIACAO[0]
print("Pergunta", EXEMPLO_QUERY["query"])
print("Documento de referência", EXEMPLO_QUERY["document_id"])
RESPOSTA_BM25 = buscar("BM25", EXEMPLO_QUERY["query"])
tabela_busca(RESPOSTA_BM25)

## 10. Busca semântica por embeddings

A busca semântica representa a pergunta como um vetor e recupera chunks próximos no espaço vetorial. Ela pode identificar conteúdos relacionados mesmo quando não existe coincidência literal de todas as palavras.

O embedding da pergunta é produzido pelo serviço de embeddings configurado no YAML. A seleção kNN ocorre no Elasticsearch.

In [ ]:
RESPOSTA_SEMANTICA = buscar("Semântica", EXEMPLO_QUERY["query"])
tabela_busca(RESPOSTA_SEMANTICA)

## 11. Busca híbrida com RRF

A busca híbrida executa recuperação lexical e vetorial. O Elasticsearch executa as duas buscas. O WebSensors Flow RAG combina os rankings por **Reciprocal Rank Fusion**, conhecido como RRF.

A fusão utiliza a posição de cada candidato nas listas. Não é necessário colocar o score do BM25 e o score vetorial na mesma escala. Nenhum modelo adicional de reranking participa desta etapa.

In [ ]:
RESPOSTA_HIBRIDA = buscar("Híbrida", EXEMPLO_QUERY["query"])
tabela_busca(RESPOSTA_HIBRIDA)
print("Informações da recuperação", RESPOSTA_HIBRIDA.get("debug"))

## 12. Comparação visual dos rankings

A tabela reúne as primeiras posições dos três métodos para a mesma pergunta. Os resultados da API são chunks, por isso um documento pode aparecer mais de uma vez. Nesta comparação a posição de cada documento corresponde à primeira ocorrência de um de seus chunks.

In [ ]:
RESPOSTAS_EXEMPLO = {
    "BM25": RESPOSTA_BM25,
    "Semântica": RESPOSTA_SEMANTICA,
    "Híbrida": RESPOSTA_HIBRIDA,
}


def ids_unicos(resposta, limite=None):
    ids = []
    encontrados = set()
    for hit in resposta.get("results", []):
        doc_id = hit.get("document_id")
        if doc_id and doc_id not in encontrados:
            ids.append(doc_id)
            encontrados.add(doc_id)
        if limite and len(ids) >= limite:
            break
    return ids


comparacao = {}
for metodo, resposta in RESPOSTAS_EXEMPLO.items():
    for posicao, doc_id in enumerate(ids_unicos(resposta, K_AVALIACAO), start=1):
        comparacao.setdefault(doc_id, {})[metodo] = posicao

quadro = pd.DataFrame.from_dict(comparacao, orient="index")
quadro.index.name = "document_id"
quadro["relevante"] = quadro.index == EXEMPLO_QUERY["document_id"]
display(quadro.sort_values(["Híbrida", "BM25"], na_position="last"))

## 13. Gabarito para avaliação da recuperação

As perguntas já possuem um texto de apoio associado no conjunto Pirá. Essa associação funciona como um julgamento de relevância positivo para recuperação documental.

O procedimento avalia a capacidade de recuperar o texto de apoio registrado no conjunto. Outros textos também podem responder à pergunta, mas não recebem automaticamente uma anotação positiva. Portanto, os números representam uma avaliação didática deste recorte e não substituem os resultados oficiais do benchmark completo.

In [ ]:
GABARITO = defaultdict(set)
for exemplo in PERGUNTAS:
    if exemplo["document_id"] in DOCUMENTOS_OK:
        GABARITO[exemplo["query_id"]].add(exemplo["document_id"])

AVALIACAO = [
    pergunta for pergunta in PERGUNTAS_AVALIACAO
    if GABARITO.get(pergunta["query_id"])
]


def recall_em_k(resultado, relevantes, k):
    selecionados = set(ids_unicos(resultado, k))
    return len(selecionados & relevantes) / len(relevantes) if relevantes else float("nan")


def rr_em_k(resultado, relevantes, k):
    return next((1.0/i for i, doc in enumerate(ids_unicos(resultado, k), start=1) if doc in relevantes), 0.0)


def ndcg_em_k(resultado, relevantes, k):
    recuperados = ids_unicos(resultado, k)
    dcg = sum(1.0 / math.log2(i + 1) for i, doc in enumerate(recuperados, start=1) if doc in relevantes)
    dcg_ideal = sum(1.0 / math.log2(i + 1) for i in range(1, min(k, len(relevantes)) + 1))
    return dcg / dcg_ideal if dcg_ideal else 0.0

print("Perguntas avaliáveis", len(AVALIACAO))
assert len(AVALIACAO) > 0

## 14. Avaliação de BM25, semântica e híbrida

Cada consulta selecionada é enviada às três modalidades. São calculadas as métricas **Recall@10**, **MRR@10** e **nDCG@10**, além da latência observada pela API.

Como a API devolve chunks, os resultados são agrupados por documento antes do cálculo. Essa distinção é importante para comparar com o gabarito, que está definido no nível de documento.

In [ ]:
REGISTROS_AVALIACAO = []
FALHAS_BUSCA = []
for pergunta in tqdm(AVALIACAO, desc="Avaliando consultas", unit="pergunta"):
    relevantes = GABARITO[pergunta["query_id"]]
    for metodo in ROTAS:
        try:
            resposta = buscar(metodo, pergunta["query"], top_k=TOP_K_API)
            REGISTROS_AVALIACAO.append({
                "query_id": pergunta["query_id"],
                "metodo": metodo,
                "recall_10": recall_em_k(resposta, relevantes, K_AVALIACAO),
                "rr_10": rr_em_k(resposta, relevantes, K_AVALIACAO),
                "ndcg_10": ndcg_em_k(resposta, relevantes, K_AVALIACAO),
                "latencia_ms": resposta["took_ms"],
                "documentos_retornados": len(ids_unicos(resposta)),
            })
        except Exception as erro:
            FALHAS_BUSCA.append({"query_id": pergunta["query_id"], "metodo": metodo, "erro": str(erro)})

DF_AVALIACAO = pd.DataFrame(REGISTROS_AVALIACAO)
if DF_AVALIACAO.empty:
    raise RuntimeError("Nenhuma consulta terminou com sucesso")
RESUMO = (DF_AVALIACAO.groupby("metodo", as_index=False)
          .agg(
              consultas=("query_id", "nunique"),
              recall_10=("recall_10", "mean"),
              mrr_10=("rr_10", "mean"),
              ndcg_10=("ndcg_10", "mean"),
              latencia_media_ms=("latencia_ms", "mean"),
          ))
display(RESUMO.round(4))
print("Consultas com falha", len(FALHAS_BUSCA))
if FALHAS_BUSCA:
    display(pd.DataFrame(FALHAS_BUSCA).head(10))

## 15. Gráficos de qualidade e latência

Os gráficos ajudam a visualizar a diferença entre os métodos no conjunto selecionado. Qualidade e tempo são apresentados separadamente, pois representam critérios diferentes de avaliação.

In [ ]:
fig, eixo = plt.subplots(figsize=(9, 4.8))
RESUMO.set_index("metodo")[["recall_10", "mrr_10", "ndcg_10"]].plot.bar(ax=eixo, rot=0)
eixo.set_ylabel("Valor médio")
eixo.set_xlabel("Método")
eixo.set_ylim(0, 1.05)
eixo.set_title("Qualidade de recuperação em português")
eixo.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

fig, eixo = plt.subplots(figsize=(8, 4.2))
RESUMO.set_index("metodo")["latencia_media_ms"].plot.bar(ax=eixo, rot=0)
eixo.set_ylabel("Milissegundos")
eixo.set_xlabel("Método")
eixo.set_title("Latência média observada pela API")
eixo.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 16. Consultas livres em português

Além das perguntas anotadas, a coleção pode receber novas perguntas formuladas pelo usuário. Esta célula permite comparar os três métodos usando uma consulta livre.

Os documentos podem conter informações científicas sobre ecossistemas costeiros, biodiversidade e conservação marinha.

In [ ]:
CONSULTA_LIVRE = "Quais fatores ambientais afetam a biodiversidade de organismos marinhos?"
for metodo in ROTAS:
    print("\nMétodo", metodo)
    resultado = buscar(metodo, CONSULTA_LIVRE)
    tabela_busca(resultado, limite=5)

## 17. Consulta de documentos e materiais gerados

O serviço mantém metadados sobre as revisões indexadas. A rota de documentos permite consultar o registro de um arquivo e encontrar as referências aos artefatos Markdown e JSON no MinIO Silver.

In [ ]:
CATALOGO = chamar_api("GET", "/documents", parametros={"limit": 200})
DOCUMENTOS_BENCHMARK = [
    item for item in CATALOGO.get("documents", [])
    if item.get("metadata", {}).get("colecao") == COLECAO
]
print("Documentos da coleção presentes no catálogo", len(DOCUMENTOS_BENCHMARK))
if DOCUMENTOS_BENCHMARK:
    exemplo_catalogo = DOCUMENTOS_BENCHMARK[0]
    detalhamento = chamar_api("GET", f"/documents/{exemplo_catalogo['document_id']}")
    print(json.dumps({
        "document_id": detalhamento.get("document_id"),
        "status": detalhamento.get("status"),
        "silver": detalhamento.get("silver"),
    }, ensure_ascii=False, indent=2))

## 18. Exportação dos resultados

O resumo por método e os resultados individuais podem ser salvos para inspeção posterior. A exportação mantém os identificadores das perguntas, as métricas e a latência, sem reproduzir o conteúdo documental.

In [ ]:
PASTA_SAIDAS = Path("resultados_pira2")
PASTA_SAIDAS.mkdir(exist_ok=True)
DF_AVALIACAO.to_csv(PASTA_SAIDAS / "resultados_consultas.csv", index=False)
RESUMO.to_csv(PASTA_SAIDAS / "resumo_metodos.csv", index=False)
(PASTA_SAIDAS / "configuracao.json").write_text(
    json.dumps({
        "dataset": "paulopirozelli/pira",
        "split": DIVISAO,
        "colecao": COLECAO,
        "documentos_indexados": len(DOCUMENTOS_OK),
        "consultas_avaliadas": len(AVALIACAO),
        "limite_documentos": MAX_DOCUMENTOS,
        "k_avaliacao": K_AVALIACAO,
        "top_k_api": TOP_K_API,
        "semente": SEMENTE,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Resultados salvos em", PASTA_SAIDAS.resolve())

## 19. Interpretação dos resultados

O BM25 favorece correspondências exatas de vocabulário. A busca semântica permite recuperar textos relacionados ao significado da pergunta. A busca híbrida combina as posições dos candidatos de ambas as estratégias pelo RRF implementado pelo WebSensors Flow RAG.

Um resultado híbrido não é necessariamente superior em todas as perguntas. A comparação deve considerar as métricas, o tamanho da coleção, os parâmetros do Elasticsearch, os embeddings usados e a presença de múltiplos chunks para o mesmo documento.

O Pirá 2.0 disponibiliza textos e perguntas associados. Este notebook transforma essas associações em gabarito documental para uma avaliação limitada à coleção indexada. Assim, os valores obtidos não devem ser comparados diretamente aos números de estudos que utilizam outra seleção de documentos, outra granularidade ou o protocolo oficial completo do benchmark.

Referência acadêmica no [artigo sobre os benchmarks do Pirá 2.0](https://doi.org/10.1162/dint_a_00245).

In [ ]:
api.close()
print("Experimento concluído")